# Validation audit

Run an existing stacked BiLSTM checkpoint on full VAL and labeling_sample. Set the checkpoint and its original tokenizer below. Use the same cleanup options as the submitted run. An optional submission CSV checks whether TEST predictions reproduce that file.

## Setup

In [8]:
import os
import sys
import base64
import subprocess
from pathlib import Path

repo_dir = Path("/content/UA-Locations-NER-Deep-Learning")
if not repo_dir.exists():
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    env = dict(os.environ, GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.extraheader",
               GIT_CONFIG_VALUE_0=f"AUTHORIZATION: basic {auth}")
    try:
        subprocess.run(["git", "clone",
                        "https://github.com/dkalchenko01/UA-Locations-NER-Deep-Learning.git",
                        str(repo_dir)], env=env, check=True)
    finally:
        del token, auth, env
sys.path.insert(0, str(repo_dir))


In [9]:
%pip install -q -r "$repo_dir/requirements.txt"

import json
import pandas as pd
import torch
from google.colab import drive
from tokenizers import Tokenizer

from src.data_io import get_data_dir, load_manifest, load_data_splits, file_sha256, save_json
from src.dataset import prepare_ner_records, prepare_inference_records, make_loader
from src.model import BiLSTMTagger
from src.training import predict
from src.metrics import competition_scores, location_errors
from src.postprocessing import postprocess
from src.preprocessing import clean_locations_v3

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Checkpoint and submission settings

Replace both paths. `tokenizer_path` must point to the tokenizer used for this checkpoint. Set `submission_path` to an existing submission CSV to check exact TEST reproduction, or keep `None`. Expansion below applies to spans containing more than four letters.

In [10]:
checkpoint_path = Path("/content/drive/MyDrive/UA-Locations-NER/stacked/stacked_20261112T084900Z_7c1467a9/epoch_03.pt")
tokenizer_path = repo_dir / "artifacts/tokenizers/bpe_bytelevel.json"
submission_path = None
clean_options = {"expand": True, "trim": True, "empty": True}

splits_dir = repo_dir / "data/splits"
output_dir = checkpoint_path.parent / f"audit_{checkpoint_path.stem}"
output_dir.mkdir(exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
config = checkpoint["config"]
provenance = checkpoint.get("provenance", {})
for path, key in [(tokenizer_path, "tokenizer_sha256"),
                  (splits_dir / "validation_manifest.parquet", "manifest_sha256")]:
    if provenance.get(key) and file_sha256(path) != provenance[key]:
        raise ValueError(f"Checkpoint provenance mismatch: {path}")
tokenizer = Tokenizer.from_file(str(tokenizer_path))
tokenizer.no_padding()
tokenizer.no_truncation()
model = BiLSTMTagger(**checkpoint["model_config"]).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
loader_options = {"pad_id": tokenizer.token_to_id("[PAD]"),
                  "batch_size": config["batch_size"],
                  "max_padded_tokens": config["max_padded_tokens"]}
print(device, checkpoint_path, "epoch", checkpoint["epoch"])


cpu /content/drive/MyDrive/UA-Locations-NER/stacked/stacked_20261112T084900Z_7c1467a9/epoch_03.pt epoch 3


## Full validation and labeling sample

Keep every validation row, including examples without aligned BIO labels. These examples contribute to F1 and are excluded from token loss.

In [11]:
data_dir = get_data_dir("colab")
manifest = load_manifest(splits_dir)
val_records, alignment = [], []
for source in config["sources"]:
    splits = load_data_splits("colab", splits_dir, sources=(source,))
    records, skipped = prepare_ner_records(tokenizer, splits[f"{source}_val"], source, "val")
    val_records.extend(records)
    alignment.append({"source": source, "rows": len(records), "unaligned": len(skipped)})
    del splits
expected = manifest[
    (manifest["split"] == "val") & manifest["source"].isin(config["sources"])]
assert {r["text_id"] for r in val_records} == {
    f"{row.source}_{row.row_id}" for row in expected.itertuples()}
sample = pd.read_csv(data_dir / "labeling_sample.csv")
sample_records, _ = prepare_ner_records(tokenizer, sample, "sample", "val")
display(pd.DataFrame(alignment))


Prepare uk/val:   0%|          | 0/51 [00:00<?, ?it/s]

Prepare ru/val:   0%|          | 0/13 [00:00<?, ?it/s]

Prepare sample/val:   0%|          | 0/1 [00:00<?, ?it/s]

,source,rows,unaligned
0,uk,100741,165
1,ru,24626,23


## Compare decoding and cleanup

`raw` uses decoded spans, `clean_v3` applies the previous cleaner, and `submission` uses the options above. All scores use the existing competition metric. Token loss is independent of decoding and cleanup.

In [12]:
datasets = {"val": val_records, "sample": sample_records}
scores, losses, sample_predictions = [], {}, {}

def cleaned(records, predictions, mode):
    result = {}
    for record in records:
        prediction = predictions[record["text_id"]]
        locations = prediction["locations"]
        if mode == "clean_v3":
            locations = clean_locations_v3(locations)
        if mode == "submission":
            locations = postprocess(record["text"], prediction["spans"], **clean_options)
        result[record["text_id"]] = {"locations": locations}
    return result

for name, records in datasets.items():
    for decoder in ("argmax", "viterbi_lines"):
        predictions, token_losses = predict(
            model, make_loader(records, **loader_options), device, decoding=decoder)
        losses[name] = token_losses
        for mode in ("raw", "clean_v3", "submission"):
            scores.append({"dataset": name, "decoder": decoder, "cleanup": mode,
                           **competition_scores(records, cleaned(records, predictions, mode))})
        if name == "sample":
            sample_predictions[decoder] = predictions
        if name == "val" and decoder == "viterbi_lines":
            val_predictions = cleaned(records, predictions, "submission")
        del predictions
scores = pd.DataFrame(scores)
print("VAL")
display(scores.loc[scores["dataset"] == "val",
                   ["decoder", "cleanup", "uk_comp_f1", "ru_comp_f1", "combined_comp_f1"]].round(4))
print("Labeling sample")
display(scores.loc[scores["dataset"] == "sample",
                   ["decoder", "cleanup", "combined_comp_f1"]]
        .rename(columns={"combined_comp_f1": "f1"}).round(4))
scores.to_csv(output_dir / "scores.csv", index=False)
save_json(output_dir / "settings.json", {
    "checkpoint": str(checkpoint_path), "epoch": checkpoint["epoch"],
    "checkpoint_sha256": file_sha256(checkpoint_path),
    "tokenizer_sha256": file_sha256(tokenizer_path),
    "manifest_sha256": file_sha256(splits_dir / "validation_manifest.parquet"),
    "cleanup": clean_options, "token_losses": losses})


Predict:   0%|          | 0/986 [00:00<?, ?it/s]

Predict:   0%|          | 0/986 [00:00<?, ?it/s]

Predict:   0%|          | 0/1 [00:00<?, ?it/s]

Predict:   0%|          | 0/1 [00:00<?, ?it/s]

VAL


,decoder,cleanup,uk_comp_f1,ru_comp_f1,combined_comp_f1
0,argmax,raw,0.8865,0.9117,0.8928
1,argmax,clean_v3,0.8861,0.9117,0.8926
2,argmax,submission,0.8813,0.9170,0.8904
3,viterbi_lines,raw,0.8903,0.9178,0.8973
4,viterbi_lines,clean_v3,0.8899,0.9178,0.8970
5,viterbi_lines,submission,0.8843,0.9207,0.8935


Labeling sample


,decoder,cleanup,f1
6,argmax,raw,0.4842
7,argmax,clean_v3,0.5747
8,argmax,submission,0.6207
9,viterbi_lines,raw,0.5854
10,viterbi_lines,clean_v3,0.5926
11,viterbi_lines,submission,0.5926


## Validation slices

Uses cached viterbi_lines predictions with submission cleanup. Run the comparison cell above once to retain them. Slices overlap and are reported for combined, UK and RU. Length means Python characters. Location counts use gold span occurrences. Address detection is a heuristic over gold location strings using street/address keywords. Empty slices have no F1. On the zero-location slice, messages_with_predictions counts false-positive messages; F1 alone does not describe their frequency.

In [13]:
import re
from src.metrics import comp_metric

address_pattern = r"(?i)\b(?:вул\b|улиц|просп|пров\b|переул|бульв|набереж|шосе|площ|пл\.|узвіз|спуск)"
frame = pd.DataFrame([{"source": r["source"], "chars": len(r["text"]),
                       "multiline": "\n" in r["text"] or "\r" in r["text"],
                       "gold": r["gold_locations"],
                       "predicted": val_predictions[r["text_id"]]["locations"],
                       "address": any(re.search(address_pattern, s) for s in r["gold_locations"])}
                      for r in val_records])
frame["locations"] = frame["gold"].str.len()
slices = {"all": frame["chars"] >= 0,
          "chars <=100": frame["chars"] <= 100,
          "chars 101-200": frame["chars"].between(101, 200),
          "chars 201-400": frame["chars"].between(201, 400),
          "chars >400": frame["chars"] > 400,
          "multiline": frame["multiline"],
          "0 locations": frame["locations"] == 0,
          "1 location": frame["locations"] == 1,
          "2+ locations": frame["locations"] >= 2,
          "5+ locations": frame["locations"] >= 5,
          "address": frame["address"],
          "no address": ~frame["address"]}
slice_scores = []
for source in ["combined", *config["sources"]]:
    for name, mask in slices.items():
        selected = frame.loc[mask & ((frame["source"] == source) | (source == "combined"))]
        slice_scores.append({"source": source, "slice": name, "messages": len(selected),
                             "gold_spans": selected["locations"].sum(),
                             "predicted_locations": selected["predicted"].str.len().sum(),
                             "messages_with_predictions": selected["predicted"].str.len().gt(0).sum(),
                             "f1": comp_metric(selected["gold"].tolist(), selected["predicted"].tolist())
                                   if len(selected) else None})
slice_scores = pd.DataFrame(slice_scores)
display(slice_scores.loc[slice_scores["slice"] != "0 locations",
                         ["source", "slice", "messages", "f1"]].round(4))
print("Messages without gold locations")
display(slice_scores.loc[slice_scores["slice"] == "0 locations",
                         ["source", "messages", "messages_with_predictions"]]
        .rename(columns={"messages_with_predictions": "false_positive_messages"}))
slice_scores.to_csv(output_dir / "validation_slices.csv", index=False)


,source,slice,messages,f1
0,combined,all,125367,0.8935
1,combined,chars <=100,67796,0.8960
2,combined,chars 101-200,50643,0.8912
3,combined,chars 201-400,6878,0.8970
4,combined,chars >400,50,0.8810
5,combined,multiline,0,NaN
7,combined,1 location,21211,0.9277
8,combined,2+ locations,9659,0.8968
9,combined,5+ locations,521,0.9118
10,combined,address,412,0.8140


Messages without gold locations


,source,messages,false_positive_messages
6,combined,94497,1611
18,uk,77538,1267
30,ru,16959,344


## Read all sample errors

FP/FN use official normalization. Gold spans, decoded spans and original strings remain visible to inspect boundaries. Fill in `comment` in the saved review CSV.

In [14]:
review = []
for record in sample_records:
    text_id = record["text_id"]
    predictions = sample_predictions["viterbi_lines"][text_id]
    locations = postprocess(record["text"], predictions["spans"], **clean_options)
    errors = location_errors(record["gold_locations"], locations)
    review.append({"text_id": text_id, "text": record["text"],
                   "gold": record["gold_locations"], "raw": predictions["locations"],
                   "predicted": locations, **errors, "comment": ""})
    print(text_id, "\n", record["text"])
    print("GOLD:", record["gold_locations"])
    print("GOLD SPANS:", sample.loc[record["row_id"], "loc_markers"])
    print("PRED SPANS:", predictions["spans"], "RAW:", predictions["locations"])
    print("FINAL:", locations, "\nFP:", errors["false_positives"],
          "\nFN:", errors["false_negatives"], "\n")
pd.DataFrame(review).to_csv(output_dir / "sample_review.csv", index=False)


sample_0 
 -261.310
GOLD: []
GOLD SPANS: []
PRED SPANS: [] RAW: []
FINAL: [] 
FP: [] 
FN: [] 

sample_1 
 Проспект Бандери стоїть, як і пів міста… довго ще??

Життя🫶Оболонь
GOLD: ['Проспект Бандери']
GOLD SPANS: [(0, 16)]
PRED SPANS: [(53, 59)] RAW: ['Життя🫶']
FINAL: ['Життя'] 
FP: ['життя'] 
FN: ['проспект бандери'] 

sample_2 
 ☢️Радіаційний фон (мкР/год) по Київщині станом на 9 год 28 серпня:

Чорнобиль - 19
Київ - 12
Тетерів - 9
Фастів - 11
Біла Церква - 13
Миронівка - 13
Бориспіль - 13
Яготин - 12
Баришівка – 10

Всі показники у межах норми ✅

ℹ️ Нагадуємо, що допустиме значення рівня радіаційного фону до 25 мкР/год

Сайт💧Instagram💧Facebook💧YouTube
GOLD: ['Київщині', 'Чорнобиль', 'Київ', 'Тетерів', 'Фастів', 'Біла Церква', 'Миронівка', 'Бориспіль', 'Яготин', 'Баришівка']
GOLD SPANS: [(31, 39), (68, 77), (83, 87), (93, 100), (105, 111), (117, 128), (134, 143), (149, 158), (164, 170), (176, 185)]
PRED SPANS: [(31, 39), (83, 87), (117, 128), (164, 170), (176, 185)] RAW: ['Київщині', 